In [1]:
# Figure1(A-D)
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pylab as plt
from matplotlib.font_manager import FontProperties
import pyscisci.all as pyscisci
from brokenaxes import brokenaxes
from matplotlib import gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.ticker import LogLocator
from matplotlib.ticker import MultipleLocator
import matplotlib.ticker as ticker
from scipy import stats
from scipy.stats import norm, ks_2samp
from adjustText import adjust_text
# 定义字体
EF = FontProperties(family = 'DejaVu Sans')

%matplotlib inline

In [2]:
main_path = r'/home/20250114zmz_kd/'

In [3]:
basic_performance = r'Co-utilization/2-Revision202501-DataVisualize/2-BasicPerformanceGaps.csv'
bp_df = pd.read_csv(main_path + basic_performance)
del bp_df['Unnamed: 0']
print(bp_df .shape)
bp_df .columns

(212735, 14)


Index(['work_id', 'PublishedYear', 'numfacility', 'C1N0', 'Facility',
       'times_cited_3', 'times_cited_5', 'times_cited_10', 'num_reference',
       'di3', 'di5', 'di10', 'NoveltyScore', 'ConventionalityScore'],
      dtype='object')

In [4]:
co_ut = bp_df[(bp_df['C1N0']==1)|((bp_df['C1N0']==0))]
print(co_ut .shape)

(212735, 14)


In [5]:
# 是否联用的绩效差异
c1 = co_ut[['work_id','PublishedYear','numfacility','num_reference',
              'times_cited_3', 'times_cited_5', 'times_cited_10',
              'di3', 'di5', 'di10', 'NoveltyScore', 'ConventionalityScore']]
c1 .shape

(212735, 12)

In [6]:
c1_tc3 = c1[['work_id','PublishedYear','numfacility','times_cited_3']]
c1_tc3 = c1_tc3[c1_tc3['PublishedYear']<=2021]
print(c1_tc3 .shape)
c1_tc3 = c1_tc3.dropna()
print(c1_tc3 .shape)

(185663, 4)
(185663, 4)


In [7]:
c1_tc5 = c1[['work_id','PublishedYear','numfacility','times_cited_5']]
c1_tc5 = c1_tc5[c1_tc5['PublishedYear']<=2019]
print(c1_tc5 .shape)
c1_tc5 = c1_tc5.dropna()
print(c1_tc5 .shape)

(156562, 4)
(156562, 4)


In [8]:
c1_di3 = c1[['work_id','PublishedYear','numfacility','times_cited_3','num_reference','di3']]
c1_di3 = c1_di3[(c1_di3['PublishedYear']<=2021)&(c1_di3['times_cited_3']>=5)&(c1_di3['num_reference']>=5)]
print(c1_di3 .shape)
c1_di3 = c1_di3.dropna()
print(c1_di3 .shape)

(132749, 6)
(132749, 6)


In [9]:
c1_di3['D1N0'] = c1_di3.apply(lambda row: 1 if row['di3']>=0 else 0, axis = 1)
c1_di3['D1N0'].value_counts()

D1N0
1    92735
0    40014
Name: count, dtype: int64

In [10]:
c1_di5 = c1[['work_id','PublishedYear','numfacility','times_cited_5','num_reference','di5']]
c1_di5 = c1_di5[(c1_di5['PublishedYear']<=2019)&(c1_di5['times_cited_5']>=5)&(c1_di5['num_reference']>=5)]
print(c1_di5 .shape)
c1_di5 = c1_di5.dropna()
print(c1_di5 .shape)

(124266, 6)
(124266, 6)


In [11]:
c1_di5['D1N0'] = c1_di5.apply(lambda row: 1 if row['di5']>=0 else 0, axis = 1)
c1_di5['D1N0'].value_counts()

D1N0
1    68290
0    55976
Name: count, dtype: int64

In [ ]:
fig, axes = plt.subplots(1,2, figsize = (9,4), dpi = 300)

sns.pointplot(data = c1_tc3, x = 'numfacility', y = 'times_cited_3', 
              errorbar=('ci', 95), markers='o', color='#e31a1c', err_kws={'linewidth': 1},
              markersize = 5, label = '3-year', capsize = 0.1,
              ax = axes[0])
sns.pointplot(data = c1_tc5, x = 'numfacility', y = 'times_cited_5', 
              errorbar=('ci', 95), markers='s', color='#e6ab02', err_kws={'linewidth': 1},
              markersize = 5, label = '5-year', capsize = 0.1,
              ax = axes[0])


axes[0].set_ylabel('Scientific Impacts', fontsize=10, fontproperties=EF, color='#1a1a1a')
axes[0].set_xlabel('Number of Facility', fontsize=10, fontproperties=EF, color='#1a1a1a')
axes[0].legend(frameon=False, prop={'family': EF.get_name(), 'size': 10}, loc='upper left')
axes[0].set_ylim(0, 300)
axes[0].yaxis.set_major_locator(MultipleLocator(60))

sns.pointplot(data = c1_di3, x = 'numfacility', y = 'D1N0', 
              errorbar=('ci', 95), markers='o', color='#e31a1c', err_kws={'linewidth': 1},
              markersize = 5, label = '3-year', capsize = 0.1,
              ax = axes[1])
sns.pointplot(data = c1_di5, x = 'numfacility', y = 'D1N0', 
              errorbar=('ci', 95), markers='s', color='#e6ab02', err_kws={'linewidth': 1},
              markersize = 5, label = '5-year', capsize = 0.1,
              ax = axes[1])

axes[1].set_ylabel('Probability of Scientific Disruption', fontsize=10, fontproperties=EF, color='#1a1a1a')
axes[1].set_xlabel('Number of Facility', fontsize=10, fontproperties=EF, color='#1a1a1a')
axes[1].legend(frameon=False, prop={'family': EF.get_name(), 'size': 10}, loc='upper left')
axes[1].set_ylim(-0.05, 1.05)
axes[1].yaxis.set_major_locator(MultipleLocator(0.2))


for i, ax in enumerate(axes.flat):
    ax.text(0.99, 1.01, chr(97 + i), transform=ax.transAxes, 
            fontsize=14, verticalalignment='bottom', horizontalalignment='center')
    
plt.tight_layout()
# plt.savefig(main_path + r'Co-utilization/2-Revision202501-DataVisualize/3-Figure-6.svg')
plt.show()